In [1]:
# injected for headless run (no stdin): scripted answers, key prompts -> ''
_menu = []
_resume = []
_topic = ''
def input(prompt=''):
    p = str(prompt)
    if 'KEY' in p.upper(): ans = ''
    elif 'Pick a demo' in p: ans = _menu.pop(0) if _menu else 'q'
    elif 'topic' in p.lower(): ans = _topic
    elif p.startswith('resume>'): ans = _resume.pop(0) if _resume else '{"type": "accept"}'
    else: ans = ''
    print(f'{p.strip()[-60:]} <auto: {ans!r}>')
    return ans


# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH05/ch05_product_reliability_rule.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# About this notebook

This notebook makes the reliability math behind multi-agent systems explicit. It implements two simple models. The first follows the product reliability rule, formalized in systems engineering as [Lusser's law](https://en.wikipedia.org/wiki/Lusser%27s_law), which shows how system accuracy decays when probabilistic components are composed sequentially without validation. Applied to a multi-agent pipeline, this gives you:

$$
P(\text { system success })=\prod_{i=1}^N p_i
$$

The second model extends this baseline by introducing validation at each agent boundary, illustrating how catching errors before they propagate fundamentally changes system-level reliability. The goal is not to capture every real-world failure mode, but to provide intuition for why multi-agent systems are fragile by default and how architectural choices directly affect reliability and cost.

In [2]:
import pandas as pd
from typing import List


def reliability_df_product_rule(
    p: float,
    max_agents: int
) -> pd.DataFrame:
    """
    Generate a DataFrame showing system reliability decay
    under the product reliability rule.
    """
    data = []
    for n in range(1, max_agents + 1):
        system_p = p ** n
        data.append({
            "num_agents": n,
            "per_agent_accuracy": p,
            "system_accuracy": system_p,
            "error_rate": 1.0 - system_p
        })
    return pd.DataFrame(data)


def reliability_df_with_validation(
    p: float,
    validation_catch_rate: float,
    max_agents: int
) -> pd.DataFrame:
    """
    Generate a DataFrame showing system reliability
    with validation at each agent boundary.
    """
    p_eff = p + (1.0 - p) * validation_catch_rate

    data = []
    for n in range(1, max_agents + 1):
        system_p = p_eff ** n
        data.append({
            "num_agents": n,
            "per_agent_accuracy": p_eff,
            "system_accuracy": system_p,
            "error_rate": 1.0 - system_p
        })
    return pd.DataFrame(data)


In [3]:
p = 0.98
v = 0.9
max_agents = 10

df_product = reliability_df_product_rule(p, max_agents)
df_validation = reliability_df_with_validation(p, v, max_agents)




In [4]:
df_product


,num_agents,per_agent_accuracy,system_accuracy,error_rate
0,1,0.98,0.980000,0.020000
1,2,0.98,0.960400,0.039600
2,3,0.98,0.941192,0.058808
3,4,0.98,0.922368,0.077632
4,5,0.98,0.903921,0.096079
5,6,0.98,0.885842,0.114158
6,7,0.98,0.868126,0.131874
7,8,0.98,0.850763,0.149237
8,9,0.98,0.833748,0.166252
9,10,0.98,0.817073,0.182927


In [5]:
df_validation

,num_agents,per_agent_accuracy,system_accuracy,error_rate
0,1,0.998,0.998000,0.002000
1,2,0.998,0.996004,0.003996
2,3,0.998,0.994012,0.005988
3,4,0.998,0.992024,0.007976
4,5,0.998,0.990040,0.009960
5,6,0.998,0.988060,0.011940
6,7,0.998,0.986084,0.013916
7,8,0.998,0.984112,0.015888
8,9,0.998,0.982143,0.017857
9,10,0.998,0.980179,0.019821
